In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
sc = spark.sparkContext
covid_cases_rdd = sc.textFile("data/datasets/covid19/cases/covid_dataset_cases.csv")
covid_states_rdd = sc.textFile("data/datasets/covid19/states/covid_dataset_states.csv")

### 6. List the twitter handle and fips code for the top 15 states with the highest number of total cases.

In [3]:
### Covid Cases RDD
# ((state, fips), total)
covid_map_rdd = covid_cases_rdd.map(lambda x: (x.split(',')[1], int(x.split(',')[28])))

In [4]:
covid_map_rdd.take(5)

[('AP', 2), ('AP', 2), ('HP', 2), ('HP', 2), ('AS', 2)]

In [5]:
# Calculating top 15 states with highest number of total cases
covid_rdd_agg = covid_map_rdd.reduceByKey(lambda x,y: x+y).sortBy(lambda x: x[1], ascending=False)

In [6]:
covid_rdd_agg.take(15)

[('WA', 2100),
 ('GA', 1034),
 ('MH', 730),
 ('CA', 515),
 ('MI', 61),
 ('GJ', 35),
 ('AZ', 34),
 ('BR', 23),
 ('RI', 16),
 ('JH', 13),
 ('CG', 8),
 ('KA', 5),
 ('AP', 4),
 ('HP', 4),
 ('AS', 2)]

In [7]:
### Covid states RDD
# (state, (fips, twitter))
covid_states_map_rdd = covid_states_rdd.map(lambda x: (x.split(',')[0], (x.split(',')[8], x.split(',')[5])))

In [8]:
covid_states_map_rdd.collect()

[('HP', ('53', '@HPCovid')),
 ('AS', ('6', '@ASCovid')),
 ('HR', ('9', '@HRCovid')),
 ('KA', ('53', '@KACovid')),
 ('WA', ('44', '@WACovid')),
 ('CG', ('53', '@CGCovid')),
 ('BR', ('53', '@BRCovid')),
 ('JH', ('53', '@JHCovid')),
 ('GJ', ('44', '@GJCovid')),
 ('MH', ('26', '@MHCovid')),
 ('GA', ('44', '@GACovid')),
 ('MI', ('53', '@MICovid')),
 ('RI', ('26', '@RICovid')),
 ('AZ', ('53', '@AZCovid')),
 ('CA', ('4', '@CACovid')),
 ('TN', ('2', '@CACovid'))]

In [9]:
# (state, ((fips, twitter), total)
joined_rdd = covid_states_map_rdd.join(covid_rdd_agg)

In [10]:

joined_rdd.collect()

[('GJ', (('44', '@GJCovid'), 35)),
 ('MH', (('26', '@MHCovid'), 730)),
 ('AS', (('6', '@ASCovid'), 2)),
 ('HR', (('9', '@HRCovid'), 2)),
 ('KA', (('53', '@KACovid'), 5)),
 ('AZ', (('53', '@AZCovid'), 34)),
 ('RI', (('26', '@RICovid'), 16)),
 ('CA', (('4', '@CACovid'), 515)),
 ('HP', (('53', '@HPCovid'), 4)),
 ('CG', (('53', '@CGCovid'), 8)),
 ('BR', (('53', '@BRCovid'), 23)),
 ('JH', (('53', '@JHCovid'), 13)),
 ('WA', (('44', '@WACovid'), 2100)),
 ('GA', (('44', '@GACovid'), 1034)),
 ('MI', (('53', '@MICovid'), 61))]

### Note: Join and ReduceBy are a part of PairRDDFunctions class. So it has to be a paired RDD.